In [1]:
from pathlib import Path
from PIL import Image
from collections import Counter
import hashlib
import shutil
import yaml

In [2]:
ROOT = Path.cwd()

if not (ROOT / "dataset_merged").exists():
    ROOT = ROOT.parent

SOURCE = ROOT / "dataset_merged"
OUTPUT = ROOT / "dataset_cleaned"

print("Dataset awal :", SOURCE)
print("Dataset hasil:", OUTPUT)

Dataset awal : c:\Users\LENOVO\Smart-EvaluationSensing-Statistical-System\dataset_merged
Dataset hasil: c:\Users\LENOVO\Smart-EvaluationSensing-Statistical-System\dataset_cleaned


In [3]:
splits = ["train", "valid", "test"]

for split in splits:
    (OUTPUT / split / "images").mkdir(parents=True, exist_ok=True)
    (OUTPUT / split / "labels").mkdir(parents=True, exist_ok=True)

print("Folder dataset_cleaned berhasil dibuat.")

Folder dataset_cleaned berhasil dibuat.


In [13]:
# CLEANING GAMBAR
# Batas minimum resolusi
MIN_WIDTH = 180
MIN_HEIGHT = 180

valid_extensions = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]

print("Minimum resolusi:", MIN_WIDTH, "x", MIN_HEIGHT)

corrupt_images = []
low_resolution_images = []
duplicate_images = []

seen_hashes = {}

for split in splits:
    
    image_dir = SOURCE / split / "images"
    
    for image_file in image_dir.iterdir():
        
        if image_file.suffix.lower() not in valid_extensions:
            continue
        
        # Cek gambar korup
        try:
            with Image.open(image_file) as img:
                width, height = img.size
                
                # Cek resolusi
                if width < MIN_WIDTH or height < MIN_HEIGHT:
                    low_resolution_images.append(image_file)
                    continue
                
                # Cek duplicate
                file_hash = hashlib.md5(image_file.read_bytes()).hexdigest()
                
                if file_hash in seen_hashes:
                    duplicate_images.append(image_file)
                    continue
                
                seen_hashes[file_hash] = image_file
                
        except Exception:
            corrupt_images.append(image_file)
            continue
        
        # Copy gambar valid
        destination = OUTPUT / split / "images" / image_file.name
        shutil.copy2(image_file, destination)

print("Preprocessing gambar selesai.")
print("File korup      :", len(corrupt_images))
print("Resolusi rendah :", len(low_resolution_images))
print("Duplikat        :", len(duplicate_images))

Minimum resolusi: 180 x 180
Preprocessing gambar selesai.
File korup      : 0
Resolusi rendah : 0
Duplikat        : 19


In [5]:
# VALIDASI LABEL
NUM_CLASSES = 3

invalid_labels = []
fixed_labels = []

for split in splits:
    
    label_dir = SOURCE / split / "labels"
    
    for label_file in label_dir.glob("*.txt"):
        
        valid_lines = []
        
        with open(label_file, "r") as f:
            lines = f.readlines()
        
        for line_number, line in enumerate(lines, start=1):
            
            parts = line.strip().split()
            
            # Format harus 5 nilai
            if len(parts) != 5:
                invalid_labels.append(
                    (label_file, line_number, "format")
                )
                continue
            
            try:
                class_id = int(parts[0])
                x, y, w, h = map(float, parts[1:])
                
                # Cek class ID
                if class_id < 0 or class_id >= NUM_CLASSES:
                    invalid_labels.append(
                        (label_file, line_number, "class_id")
                    )
                    continue
                
                # Cek bounding box
                if not (0 <= x <= 1 and
                        0 <= y <= 1 and
                        0 < w <= 1 and
                        0 < h <= 1):
                    invalid_labels.append(
                        (label_file, line_number, "bbox")
                    )
                    continue
                
                valid_lines.append(
                    f"{class_id} {x} {y} {w} {h}\n"
                )
                
            except ValueError:
                invalid_labels.append(
                    (label_file, line_number, "value")
                )
        
        # Jika ada label valid, simpan
        image_name = label_file.stem
        
        for split_check in splits:
            image_path = OUTPUT / split_check / "images" / f"{image_name}{label_file.suffix}"
            
            # Cari berdasarkan berbagai ekstensi
            possible_images = []
            for ext in valid_extensions:
                possible_images.append(
                    OUTPUT / split_check / "images" / f"{image_name}{ext}"
                )
            
            if any(x.exists() for x in possible_images):
                destination_label = (
                    OUTPUT / split_check / "labels" / label_file.name
                )
                
                with open(destination_label, "w") as f:
                    f.writelines(valid_lines)
                
                break

print("Label invalid:", len(invalid_labels))

Label invalid: 0


In [6]:
# MATCHING LABEL WITH GAMBAR
missing_labels = []
missing_images = []

for split in splits:
    
    image_dir = OUTPUT / split / "images"
    label_dir = OUTPUT / split / "labels"
    
    images = list(image_dir.glob("*"))
    
    for image in images:
        if image.suffix.lower() not in valid_extensions:
            continue
        
        label = label_dir / f"{image.stem}.txt"
        
        if not label.exists():
            missing_labels.append(image)
    
    for label in label_dir.glob("*.txt"):
        
        image_exists = False
        
        for ext in valid_extensions:
            if (image_dir / f"{label.stem}{ext}").exists():
                image_exists = True
                break
        
        if not image_exists:
            missing_images.append(label)

print("Gambar tanpa label :", len(missing_labels))
print("Label tanpa gambar :", len(missing_images))

Gambar tanpa label : 0
Label tanpa gambar : 0


In [8]:
# Baca data.yaml dari dataset awal
DATA_YAML = SOURCE / "data.yaml"

with open(DATA_YAML, "r") as f:
    data = yaml.safe_load(f)

class_names = data["names"]

# Jumlah kelas otomatis
NUM_CLASSES = len(class_names)

print("Jumlah kelas :", NUM_CLASSES)
print("Daftar kelas :", class_names)

Jumlah kelas : 3
Daftar kelas : ['space-empty', 'space-occupied', 'illegal-parking']


In [9]:
# DATA YML UNTUK HASIL PREPROCESS
clean_yaml = {
    "path": str(OUTPUT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": NUM_CLASSES,
    "names": class_names
}

with open(OUTPUT / "data.yaml", "w") as f:
    yaml.dump(clean_yaml, f, sort_keys=False)

print("data.yaml berhasil dibuat:")
print(OUTPUT / "data.yaml")

data.yaml berhasil dibuat:
c:\Users\LENOVO\Smart-EvaluationSensing-Statistical-System\dataset_cleaned\data.yaml


In [10]:
# PERBANDINGAN SEBELUM DAN SESUDAH CLEANING
def count_images(dataset_path):
    result = {}
    
    for split in splits:
        image_dir = dataset_path / split / "images"
        
        count = 0
        
        if image_dir.exists():
            for file in image_dir.iterdir():
                if file.suffix.lower() in valid_extensions:
                    count += 1
        
        result[split] = count
    
    return result


before = count_images(SOURCE)
after = count_images(OUTPUT)

print("=== PERBANDINGAN DATASET ===")

for split in splits:
    print(
        f"{split:5s}: "
        f"sebelum = {before[split]}, "
        f"sesudah = {after[split]}, "
        f"berkurang = {before[split] - after[split]}"
    )

=== PERBANDINGAN DATASET ===
train: sebelum = 9688, sesudah = 2229, berkurang = 7459
valid: sebelum = 2198, sesudah = 281, berkurang = 1917
test : sebelum = 761, sesudah = 166, berkurang = 595


In [14]:
print("=== HASIL PREPROCESSING ===")

print("\nDataset awal :", SOURCE)
print("Dataset baru :", OUTPUT)

print("\nCleaning:")
print("- File korup      :", len(corrupt_images))
print("- Duplikat        :", len(duplicate_images))
print("- Resolusi rendah :", len(low_resolution_images))
print("- Label invalid   :", len(invalid_labels))

print("\nJumlah gambar:")
for split in splits:
    print(f"- {split}: {after[split]}")

print("\nPreprocessing selesai.")

=== HASIL PREPROCESSING ===

Dataset awal : c:\Users\LENOVO\Smart-EvaluationSensing-Statistical-System\dataset_merged
Dataset baru : c:\Users\LENOVO\Smart-EvaluationSensing-Statistical-System\dataset_cleaned

Cleaning:
- File korup      : 0
- Duplikat        : 19
- Resolusi rendah : 0
- Label invalid   : 0

Jumlah gambar:
- train: 2229
- valid: 281
- test: 166

Preprocessing selesai.
